# Pokédex preparation and validation

Read-only inspection of corrected analysis snapshot. Raw CSV remains unchanged.
Rebuild explicitly with `pokeml data clean`. Current evolution labels include hindsight; community estimates belong in later experiments.

In [1]:
# Resolve project root before package imports. Works from root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "pokeml").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook inside the ml_pokemon project")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if sys.version_info[:2] != (3, 14):
    raise RuntimeError("Select project Python 3.14.8 kernel (.pokemon)")
print(f"Project: {PROJECT_ROOT} | Python: {sys.version.split()[0]} | Kernel: {sys.executable}")


Project: /Users/pano/Library/Mobile Documents/com~apple~CloudDocs/coding/portfolio/ml_pokemon | Python: 3.14.8 | Kernel: /Users/pano/venv/pokemon/.pokemon/bin/python


In [2]:
import json
import hashlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pokeml.data.validation import validate_pkdx
from pokeml.pipeline.schema import BASE_FEATURES

df = pd.read_csv("datasets/pkdx_min.csv")
clean = pd.read_csv("datasets/pkdx_clean.csv")
manifest = json.loads(Path("datasets/pkdx_manifest.json").read_text())
corrections = pd.read_csv("datasets/pkdx_corrections.csv")
display(pd.Series(validate_pkdx(df), name="validation"))
display(df.head())

rows        1082
species     1025
families     540
valid       True
Name: validation, dtype: object

,name,type_1,type_2,rarity,stage,shape,color,total_stats,height,weight,id,species_id,evolution_chain_id,generation
0,bulbasaur,grass,poison,regular,s1c3,quadruped,green,318,0.7,6.9,1,1,1,1
1,ivysaur,grass,poison,regular,s2c3,quadruped,green,405,1.0,13.0,2,2,1,1
2,venusaur,grass,poison,regular,s3c3,quadruped,green,525,2.0,100.0,3,3,1,1
3,charmander,fire,NaN,regular,s1c3,upright,red,309,0.6,8.5,4,4,2,1
4,charmeleon,fire,NaN,regular,s2c3,upright,red,405,1.1,19.0,5,5,2,1


## Provenance and schema

Species and family IDs support dependence checks. Generation supports forward validation. None enters model inputs. Heights use metres; weights use kilograms.

In [3]:
for path, expected in manifest["files"].items():
    assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == expected, path
print("All snapshot hashes match manifest")
print("Reference commit:", manifest["reference"]["commit"])
print("Model inputs:", BASE_FEATURES)
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "unique": df.nunique()}))

All snapshot hashes match manifest
Reference commit: c80757193bd0889054e36f4209762360bdaa4b95
Model inputs: ('type_1', 'type_2', 'rarity', 'stage', 'shape', 'color', 'height', 'weight')


,dtype,missing,unique
name,str,0,1082
type_1,str,0,18
type_2,str,509,18
rarity,str,0,2
stage,str,0,6
shape,str,0,14
color,str,0,10
total_stats,int64,0,207
height,float64,0,56
weight,float64,0,496


## Corrections

Show stage changes separately from harmless name separator changes. Regional forms use introduction generation. Ambiguous ancestor chain lengths remain explicitly documented.

In [4]:
stage_changes = corrections[corrections.raw_stage != corrections.stage]
display(stage_changes)
display(corrections[corrections.raw_generation != corrections.generation])
print("Stage corrections versus raw:", len(stage_changes))
display(clean.loc[clean.name.isin(["darmanitan", "darmanitan_galar_zen", "dialga_origin", "palkia_origin"]), ["id", "name", "species_id", "generation", "stage", "total_stats"]])

,id,raw_name,name,raw_stage,stage,raw_generation,generation,stage_source
0,29,nidoran-f,nidoran_f,single,s1c3,1,1,species_branch
1,32,nidoran-m,nidoran_m,single,s1c3,1,1,species_branch
2,83,farfetchd,farfetchd,s1c2,single,1,1,form_route
3,122,mr-mime,mr_mime,single,s2c2,1,1,form_route
4,211,qwilfish,qwilfish,s1c2,single,2,2,form_route
5,222,corsola,corsola,s1c2,single,2,2,form_route
7,263,zigzagoon,zigzagoon,s1c3,s1c2,3,3,form_route
8,264,linoone,linoone,s2c3,s2c2,3,3,form_route
11,439,mime-jr,mime_jr,single,s1c3,4,4,form_route
12,474,porygon-z,porygon_z,s1c3,s3c3,4,4,species_branch


,id,raw_name,name,raw_stage,stage,raw_generation,generation,stage_source
87,10091,rattata-alola,rattata_alola,s1c2,s1c2,1,7,species_branch
88,10092,raticate-alola,raticate_alola,s2c2,s2c2,1,7,species_branch
89,10100,raichu-alola,raichu_alola,s3c3,s3c3,1,7,species_branch
90,10101,sandshrew-alola,sandshrew_alola,s1c2,s1c2,1,7,species_branch
91,10102,sandslash-alola,sandslash_alola,s2c2,s2c2,1,7,species_branch
92,10103,vulpix-alola,vulpix_alola,s1c2,s1c2,1,7,species_branch
93,10104,ninetales-alola,ninetales_alola,s2c2,s2c2,1,7,species_branch
94,10105,diglett-alola,diglett_alola,s1c2,s1c2,1,7,species_branch
95,10106,dugtrio-alola,dugtrio_alola,s2c2,s2c2,1,7,species_branch
96,10107,meowth-alola,meowth_alola,s1c2,s1c2,1,7,species_branch


Stage corrections versus raw: 23


,id,name,species_id,generation,stage,total_stats
554,555,darmanitan,555,5,s2c2,480
1059,10178,darmanitan_galar_zen,555,8,s2c2,540
1078,10245,dialga_origin,483,8,single,680
1079,10246,palkia_origin,484,8,single,680


## Analysis populations

Counts include selected forms, which are not independent species. Stage distributions establish candidate baselines; they do not establish predictive accuracy.

In [5]:
display(pd.crosstab(df.stage, df.rarity))
display(df.groupby("generation").agg(rows=("id", "size"), species=("species_id", "nunique"), median_bst=("total_stats", "median")))
stage_stats = df.groupby(["rarity", "stage"]).total_stats.agg(["count", "median", "mean", "std", "min", "max"])
display(stage_stats.round(2))

rarity,legendary,regular
stage,,
s1c2,3,237
s1c3,1,116
s2c2,3,264
s2c3,1,118
s3c3,2,125
single,95,117


,rows,species,median_bst
generation,,,
1,151,151,405.0
2,100,100,415.0
3,135,135,410.0
4,107,107,480.0
5,156,156,445.5
6,72,72,450.0
7,106,106,475.5
8,133,133,483.0
9,122,122,488.5


count  median    mean     std  min  max
rarity    stage                                          
legendary s1c2        3   385.0  406.33  118.45  300  534
          s1c3        1   200.0  200.00     NaN  200  200
          s2c2        3   570.0  573.33   25.17  550  600
          s2c3        1   400.0  400.00     NaN  400  400
          s3c3        2   680.0  680.00    0.00  680  680
          single     95   590.0  604.05   47.66  450  720
regular   s1c2      237   308.0  312.23   54.19  180  535
          s1c3      116   300.0  285.10   40.36  180  395
          s2c2      264   487.0  482.09   38.44  236  600
          s2c3      118   405.0  391.02   59.67  205  515
          s3c3      125   523.0  518.50   42.94  385  670
          single    117   478.0  476.88   70.20  175  590

In [6]:
from pokeml.visualisation.viz_eda.stage_buckets import plot_stage_buckets
fig, ax = plot_stage_buckets(df)
plt.show()

/var/folders/h8/kn6p4vt50_v066v_y6f9qp9w0000gn/T/ipykernel_16159/4094304781.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Next: [stage buckets](01_stage_analysis.ipynb), [potential power creep](02_power_creep.ipynb), [typing](03_typing_analysis.ipynb), then [full workflow](04_workflow.ipynb).